# Provider 설정: Stripe(Privy)

## 개요

**Amazon Bedrock AgentCore payments**는 Privy를 wallet infrastructure로 사용할 수 있습니다. Privy는 최종 사용자를 위한 embedded wallet을 provision하고 private key를 secure enclave에 저장하므로 AgentCore는 private key에 액세스할 수 없습니다.

이 가이드에서는 다음 세 단계를 안내합니다.

1. Privy app을 생성하고 App ID와 App Secret을 복사합니다.
2. authorization key를 생성하고 key ID와 private key를 복사합니다.
3. 최종 사용자가 자신을 대신해 sign할 권한을 에이전트에 부여할 수 있도록 Privy reference frontend를 설정합니다.

### 준비되는 항목

이 가이드를 마치면 `.env`에 추가할 네 가지 값과 `http://localhost:3000`에서 실행되는 Privy reference frontend가 준비됩니다.

| Variable | 설명 |
|:---------|:------------|
| `PRIVY_APP_ID` | Privy app 식별 |
| `PRIVY_APP_SECRET` | Privy API 호출 인증 |
| `PRIVY_AUTHORIZATION_ID` | agent signing용 key quorum ID |
| `PRIVY_AUTHORIZATION_PRIVATE_KEY` | AgentCore가 Privy 인증에 사용하는 P-256 private key |

**참고:** Authorization Private Key는 Privy dashboard에 한 번만 표시됩니다. dialog를 닫기 전에 복사하세요.

### 사전 요구 사항

* 유효한 email address
* 로컬 시스템에 Node.js 18+ 및 git 설치(Privy reference frontend용)

### 예상 소요 시간

Privy 계정 생성, authorization key 생성, Privy reference frontend 설정에 약 15~20분이 걸립니다.

### 수행할 작업

| 단계 | 위치 | 작업 |
|:-----|:------|:-------|
| 1 | dashboard.privy.io | app 생성 → `App ID` + `App Secret` 복사 |
| 2 | dashboard.privy.io | authorization key 생성 → `ID` + `Private Key` 복사 |
| 3 | 로컬 terminal + browser | `http://localhost:3000`에서 Privy reference frontend를 실행하고 로그인 검증 |

1~2단계에서 `.env`를 채웁니다. 3단계에서는 Privy reference frontend를 설정하며, 실제 최종 사용자 동의 단계는 wallet이 생성된 후 Tutorial 00의 7b단계에서 진행합니다.

### 다음 단계

이 가이드를 완료한 후 [Tutorial 00](../setup_agentcore_payments.ipynb)으로 돌아갑니다. `.env`는 이미 설정되어 있으며 이 Notebook의 첫 번째 코드 셀에서 `CREDENTIAL_PROVIDER_TYPE=StripePrivy`를 기록합니다.

In [ ]:
# 이 Notebook의 dependency 설치(Privy API 호출용 requests, .env 처리용 python-dotenv)
!pip install -r requirements.txt --quiet

## 1단계 — Privy App 생성

> **AgentCore payments 전용 Privy app을 생성하세요.** 운영 중인 다른 Privy app과 분리해야 합니다. allowed origin, authentication method, authorization key는 모두 AgentCore payments 구성 범위에 속합니다.

1. **[dashboard.privy.io](https://dashboard.privy.io)**으로 이동하여 email로 가입하거나 로그인합니다.
2. Privy에서 보낸 confirmation code를 붙여넣어 로그인을 완료합니다.
3. dashboard에서 **New app**을 선택합니다.
4. app 이름을 입력합니다(예: `AgentCore payments Tutorial`).
5. **Create app**을 선택합니다.
6. dialog에 **App ID**와 **App Secret**이 표시됩니다. 둘 다 복사하여 다음 셀을 실행할 때 붙여넣습니다.

   ![Create app dialog](../images/00-setup-privy-app2.png)

7. **User management → Authentication**을 엽니다.
8. **Basics**에서 **Email**이 활성화되었는지 확인합니다.
9. **External wallets**까지 아래로 이동합니다.
10. **EVM wallets**와 **SVM (Solana) wallets**를 모두 활성화합니다.

이제 다음 셀을 실행합니다. 두 input prompt가 표시되면 첫 번째에는 App ID를 붙여넣고(표시됨), 두 번째에는 App Secret을 붙여넣은(숨김, 점으로 표시됨) 후 각각 Enter를 누릅니다.


In [ ]:
import getpass
import sys

sys.path.append("../..")
from utils import update_env_file

PRIVY_APP_ID = input("Privy App ID: ").strip()
PRIVY_APP_SECRET = getpass.getpass("Privy App Secret (hidden): ").strip()

assert PRIVY_APP_ID and PRIVY_APP_SECRET, "Both values are required."

update_env_file(
    "../../.env",
    {
        "CREDENTIAL_PROVIDER_TYPE": "StripePrivy",
        "PRIVY_APP_ID": PRIVY_APP_ID,
        "PRIVY_APP_SECRET": PRIVY_APP_SECRET,
    },
)

## 2단계 — Authorization Key 생성

AgentCore에서 최종 사용자를 대신해 sign하려면 P-256 authorization key가 필요합니다. Privy dashboard에서 생성합니다.

1. Privy dashboard의 왼쪽 sidebar에서 app이 선택되었는지 확인합니다.
2. **Wallet Infrastructure → Authorization**으로 이동합니다.
3. **New key**를 선택합니다.
4. dialog에 이름을 입력합니다(예: `Demo app key`).
5. **Continue**를 선택합니다. Privy가 P-256 keypair를 생성하고 **ID**와 **Private Key**를 표시합니다.
6. 두 값을 모두 복사합니다.
7. **Save and close**를 선택합니다.

![Authorization key 생성](../images/00-setup-privy-app4.png)

다음 셀을 실행하고 안내가 표시되면 값을 붙여넣습니다. 그러면 `.env`가 업데이트됩니다.

> Privy는 private key 앞에 `wallet-auth:`를 붙입니다. 다음 셀에서 이 prefix를 자동으로 제거하므로 Privy에 표시된 값을 그대로 붙여넣으세요.

In [ ]:
import getpass
from utils import save_privy_authorization_key

PRIVY_AUTHORIZATION_ID = input("Privy Authorization ID: ").strip()
PRIVY_AUTHORIZATION_PRIVATE_KEY = getpass.getpass("Privy Authorization Private Key (hidden): ").strip()

assert PRIVY_AUTHORIZATION_ID and PRIVY_AUTHORIZATION_PRIVATE_KEY, "Both values are required."

save_privy_authorization_key(
    env_path="../../.env",
    authorization_id=PRIVY_AUTHORIZATION_ID,
    authorization_private_key=PRIVY_AUTHORIZATION_PRIVATE_KEY,
)

print(f"\n  Authorization ID: {PRIVY_AUTHORIZATION_ID}")
print("  4 Privy env vars are now set in ../../.env")

## 3단계 — 로컬 시스템에 Privy Reference Frontend 설정

> ✋ **수동 작업 — 이 단계는 Notebook을 호스팅하는 시스템이 아니라 로컬 시스템에서 실행합니다.** Privy reference frontend는 `http://localhost:3000`에서 제공되어야 하는 Next.js app입니다. Privy는 exact-match allowed origin을 적용하며 개발 중에는 `localhost`에만 plain HTTP를 허용합니다. remote Jupyter host를 사용 중이라면 3단계를 laptop에서 실행하세요. production 배포 참고 사항은 3d단계에 있습니다.

이 튜토리얼에서 가장 많은 수동 작업이 필요한 단계입니다. 하위 단계를 순서대로 따르세요.

### 로컬 시스템 사전 요구 사항

| Tool | macOS / Linux | Windows |
|:---|:---|:---|
| Node.js 18+ | [nodejs.org](https://nodejs.org) 또는 `brew install node` | [nodejs.org](https://nodejs.org) installer |
| git | macOS에 사전 설치 / Linux에서는 apt 사용 | [git-scm.com](https://git-scm.com) installer |

terminal에서 `node -v`와 `git --version`을 실행하여 확인합니다.


### 3a. Privy Reference Frontend Clone

로컬 시스템에서 terminal을 열고 다음을 실행합니다.

```bash
git clone https://github.com/privy-io/aws-agentcore-sdk
cd aws-agentcore-sdk
```

이 terminal을 열어 두세요. 아래에서 몇 가지 명령을 더 실행합니다.


### 3b. Privy Reference Frontend의 `.env.local` 생성

다음 셀을 실행합니다. 실제 App ID, App Secret, Signer ID가 이미 채워진 `.env.local` 파일 본문이 출력됩니다. 출력된 내용을 복사하세요.

In [ ]:
from utils import render_frontend_env_local

env_local_body = render_frontend_env_local(
    app_id=PRIVY_APP_ID,
    app_secret=PRIVY_APP_SECRET,
    signer_id=PRIVY_AUTHORIZATION_ID,
    network_mode="testnet",
)

print("─" * 72)
print("  Copy everything between the lines into .env.local")
print("─" * 72)
print(env_local_body)
print("─" * 72)

`aws-agentcore-sdk/` 디렉터리에 `.env.local`을 생성하고 내용을 붙여넣습니다.

**macOS / Linux:**

```bash
# 여전히 aws-agentcore-sdk 내부
nano .env.local   # 또는: vim .env.local, 또는: open -a TextEdit .env.local
# 1. 내용 붙여넣기
# 2. 파일 저장
# 3. editor 닫기
```

**Windows(PowerShell):**

```powershell
# 여전히 aws-agentcore-sdk 내부
notepad .env.local
# 1. 내용 붙여넣기
# 2. 파일 저장
# 3. editor 닫기
```


### 3c. Dependency 설치 및 Dev Server 시작

Privy reference frontend는 **pnpm**을 사용합니다. 아직 없다면 한 번 설치합니다.

```bash
npm install -g pnpm
```

그런 다음 `aws-agentcore-sdk/` 디렉터리에서 다음을 실행합니다.

```bash
pnpm install
pnpm dev
```

> **`pnpm install`이 native-binding 오류(예: `Cannot find module @tailwindcss/oxide-darwin-arm64`)로 실패하면** `node_modules`를 삭제하고 다시 시도합니다: `rm -rf node_modules && pnpm install`.

server가 준비되면 terminal에 `Local: http://localhost:3000`이 표시됩니다. 이 terminal을 계속 실행해 두세요.


### 3d. Privy App에 Dev Origin 추가

Privy는 app allowlist에 없는 origin의 browser request를 거부합니다. 곧 열어 볼 `http://localhost:3000`을 추가합니다.

1. Privy dashboard에서 **Configuration → App settings → Domains**를 엽니다.
2. **Allowed origins → Web & mobile web** 아래에서 **+ Add**를 선택합니다.
3. `http://localhost:3000`을 입력합니다.
4. **Save**를 선택합니다.

![Allowed origin](../images/00-setup-privy-app-allowed-domains.png)

> **Production 참고 사항:** `http://localhost:3000`은 튜토리얼 전용입니다. Privy reference frontend를 실제 환경에 배포할 때는 Privy dashboard에서 다음 세 작업을 수행하세요.
>
> 1. production origin(`https://app.example.com` 같은 **HTTPS** URL)을 allowed-origins 목록에 추가합니다.
> 2. `http://localhost:3000`이 더 이상 필요하지 않으면 제거합니다. production app에 dev origin을 유지하면 attack surface가 넓어집니다.
> 3. production에는 **별도의 Privy app**을 사용합니다. AgentCore payments 문서에는 다음과 같이 명시되어 있습니다: *"Create a dedicated Privy app specifically for AgentCore operations. Do not reuse Privy apps that serve other purposes."* 같은 원칙이 환경 간에도 적용되므로 dev, staging, prod에서 Privy app을 공유하지 마세요.
>
> Plain HTTP는 개발 중 `localhost`에서만 허용됩니다. Privy는 그 외 모든 origin에 HTTPS를 적용합니다.


### 3e. 로그인하여 Privy Reference Frontend 작동 검증

Privy reference frontend가 올바르게 연결되었는지 검증합니다. 사용자를 대신해 sign할 권한을 에이전트에 부여하는 실제 **동의 단계**는 AgentCore가 wallet을 provision한 후 `setup_agentcore_payments.ipynb`의 **7b단계**에서 진행합니다.

1. browser에서 **[http://localhost:3000](http://localhost:3000)**을 엽니다.
2. 최종 사용자 계정으로 사용할 email을 입력합니다. 이는 개발자인 본인이 아니라 *app 사용자*를 나타냅니다. `.env`의 `LINKED_EMAIL`에 설정할 **동일한 email**을 사용하세요.
   ![Privy 로그인](../images/00-setup-privy-app6.png)
3. Privy가 해당 email로 보낸 6-digit code를 제출합니다.
   ![Email OTP](../images/00-setup-privy-app7.png)

이 Notebook의 작업은 여기까지입니다. 로그인된 화면이 표시되어야 합니다. 7b단계에서 다시 사용하므로 **이 browser tab을 열어 두세요.**

---

> **여기에는 왜 "Connect agent" 단계가 없나요?** consent flow는 최종 사용자의 Privy wallet에 AgentCore를 *additional signer*로 등록합니다. 현재 설정 단계에서는 AgentCore 측에 아직 wallet이 없습니다. 기본 Notebook의 7단계에 있는 `CreatePaymentInstrument`에서 wallet을 생성합니다. wallet이 생성되면 7b단계에서 동의 과정을 안내하며, 계속 진행하기 전에 기본 Notebook의 helper 셀에서 Privy API를 통해 signer access가 적용되었는지 검증합니다.

> **AgentCore payments 문서의 표준 표현:** signer delegation을 사용하면 developer backend가 *sign transactions on behalf of the end user*할 수 있습니다. 이는 실제 co-signing이 아니라 authorization입니다.

## 준비 완료

**[setup_agentcore_payments.ipynb](../setup_agentcore_payments.ipynb)**으로 돌아가 처음부터 실행합니다. `.env`에서 `StripePrivy`를 자동으로 가져옵니다. wallet이 provision되면 7b단계에서 **Connect agent**를 선택하므로 그때까지 `localhost:3000` tab을 열어 두세요.

계속 진행하기 전에 다음을 빠르게 확인하세요.

- [ ] `../.env` has `CREDENTIAL_PROVIDER_TYPE=StripePrivy`
- [ ] `../.env` has `PRIVY_APP_ID`, `PRIVY_APP_SECRET`, `PRIVY_AUTHORIZATION_ID`, `PRIVY_AUTHORIZATION_PRIVATE_KEY` filled in
- [ ] Credentials를 git에 commit하지 않음(`.env`가 `.gitignore`에 포함됨)
- [ ] Authorization Private Key를 안전한 위치에 저장함
- [ ] Privy app에서 Email + EVM wallets + SVM (Solana) wallets 활성화
- [ ] Privy allowed-origins 목록에 `localhost:3000`을 추가하고 Privy reference frontend를 `http://localhost:3000`에서 실행하여 end-to-end 로그인 검증